In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l9.1 Capstone

This is the victory lap. Across this course you learned each piece in isolation:
you tokenized text (u1), built the attention block and a decoder-only transformer
(u2-u4), trained and debugged it (u5), modernized the architecture (u6), adapted it
with LoRA (u7), and optimized and served it (u8). Here you wire all of it into one
continuous pipeline.

In a single notebook you will **build** a modern PocketLM, **train** it, **evaluate**
it against a baseline, **adapt** it with LoRA, **optimize** it with 4-bit quantization
and a KV cache, **serve** it behind an inference handler, and finally write the
**engineering report** that ties the whole thing together. That report is the
model-engineer deliverable: the artifact other people actually trust and reproduce.

The model stays tiny on purpose so the whole pipeline runs in seconds, but every
stage is the real shape of how decoder-only LLMs are engineered. Let us put it all
together.

## 1. Build and train

This is u1 through u5 compressed into one cell. You take the corpus, fit a
character tokenizer (u1), instantiate a modern config (RMSNorm + SwiGLU + RoPE, the
u6 upgrades), and run the AdamW training loop you debugged in u5. The model is the
decoder-only transformer you assembled block by block in u2 through u4.

The number that matters is perplexity. A model that learned nothing would be no
better than guessing uniformly across the vocabulary, so the uniform baseline is
simply the vocab size. Beating it is the first proof that everything upstream
(tokenizer, model, loss, optimizer) is actually wired correctly.

In [ ]:
import copy
import json
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss, perplexity)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2, n_head=4,
                     n_embd=64, norm="rmsnorm", mlp="swiglu", pos="rope")
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(150 if os.environ.get("POCKETLM_CI") else 500):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
ppl = perplexity(estimate_loss(model, data, 32, 16, iters=10, generator=g))
params = sum(p.numel() for p in model.parameters())
print(f"params {params}, perplexity {ppl:.2f} vs baseline {tok.vocab_size}")

## 2. Adapt with LoRA

This is u7. Instead of retraining the whole network to specialize it, you freeze the
base weights and insert small low-rank adapters, then train only those. The payoff
is in the percentage printed below: you steer the model while updating a tiny
fraction of its parameters. That is exactly why LoRA scales to models far too large
to fine-tune in full, and why a single base model can host many cheap, swappable
adapters.

In [ ]:
from pocketlm import add_lora, trainable_parameters

adapted = copy.deepcopy(model)
add_lora(adapted, rank=4)
trainable = sum(p.numel() for p in trainable_parameters(adapted))
aopt = torch.optim.AdamW(trainable_parameters(adapted), lr=1e-3)
for _ in range(40 if os.environ.get("POCKETLM_CI") else 100):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = adapted(x, y)
    aopt.zero_grad(set_to_none=True)
    loss.backward()
    aopt.step()
print(f"LoRA trainable {trainable} / {params}  ({100 * trainable / params:.1f}%)")

## 3. Optimize: quantize + KV-cache parity

This is the first half of u8, and it is two optimizations with very different risk
profiles. Quantization is a trade: squeezing weights to 4 bits shrinks the model but
can cost accuracy, so you measure the quantized perplexity and read the degradation
honestly rather than assuming it is free.

The KV cache is the opposite. It is a pure speedup that must change nothing about the
output: it reuses past keys and values during generation instead of recomputing
them. So you assert parity, confirming the incremental cached path produces the same
result as full attention to within floating-point tolerance. An optimization that
silently changes answers is a bug, not a win.

In [ ]:
from pocketlm import (fake_quantize, scaled_dot_product_attention, KVCache,
                      cached_step)

qmodel = copy.deepcopy(model)
with torch.no_grad():
    for p in qmodel.parameters():
        p.copy_(fake_quantize(p, bits=4))
q_ppl = perplexity(estimate_loss(qmodel, data, 32, 16, iters=10,
                                 generator=torch.Generator().manual_seed(0)))
qc, kc, vc = (torch.randn(1, 2, 5, 8) for _ in range(3))
full, _ = scaled_dot_product_attention(qc, kc, vc, causal=True)
cache = KVCache()
inc = torch.cat([cached_step(qc[:, :, t:t + 1], kc[:, :, t:t + 1], vc[:, :, t:t + 1], cache)
                 for t in range(5)], dim=2)
kv_parity = bool(torch.allclose(inc, full, atol=1e-5))
print(f"4-bit perplexity {q_ppl:.2f} (fp32 was {ppl:.2f}); KV-cache parity {kv_parity}")

## 4. Serve

This is the second half of u8: the model becomes a service. You wrap the trained
PocketLM behind the inference handler, which takes a request (a prompt, a token
budget, a seed) and returns a completion. This is the boundary where a research
artifact turns into something a caller can use without knowing anything about
tensors, configs, or training loops. Seeding the request keeps the output
deterministic, which is what makes the assertions later in the notebook reliable.

In [ ]:
from pocketlm import InferenceService

svc = InferenceService(model, tok)
resp = svc.handle({"prompt": "The ", "max_new_tokens": 30, "seed": 0})
print(resp["completion"])

## 5. Engineering report

This is the deliverable that makes the rest mean something. An engineering report
(an eval card) is the honest, structured record of what you built: the architecture,
the metrics measured against a stated baseline, how the model was adapted, the
optimization trade you accepted, the exact seed and corpus needed to reproduce it,
and the limitations you are not hiding. It is what lets someone else rerun your work,
compare against it, and trust the numbers. A model without this record is a demo; a
model with it is engineering.

In [ ]:
report = {
    "artifact": "PocketLM (capstone)",
    "architecture": {"norm": "rmsnorm", "mlp": "swiglu", "pos": "rope",
                     "n_layer": cfg.n_layer, "n_head": cfg.n_head, "n_embd": cfg.n_embd},
    "params": params,
    "training": {"final_perplexity": round(ppl, 2), "uniform_baseline": tok.vocab_size},
    "adaptation": {"method": "LoRA", "trainable_params": trainable,
                   "trainable_pct": round(100 * trainable / params, 2)},
    "optimization": {"quantized_bits": 4, "quantized_perplexity": round(q_ppl, 2),
                     "kv_cache_parity": kv_parity},
    "serving": "InferenceService.handle({prompt, max_new_tokens})",
    "reproducibility": {"seed": 0, "corpus": "Gutenberg #100, 1MB slice"},
    "limitations": ["char-level tokenizer", "micro scale for CI", "single corpus"],
}
print(json.dumps(report, indent=2))

That is the full arc of the course, build, train, modernize, adapt, optimize, serve,
carried out on a model small enough to run in seconds yet faithful to how real
decoder-only LLMs are engineered. The cell below is the safety net: a set of
assertions that fail loudly if any stage regressed (the model beat the baseline, the
adapter stayed small, the cache stayed exact, the service returned the right number
of tokens, and the report is complete). Green here means the whole pipeline holds
together end to end.

In [ ]:
assert report["params"] > 0
assert ppl < tok.vocab_size                      # beat the uniform baseline
assert report["adaptation"]["trainable_pct"] < 20
assert report["optimization"]["kv_cache_parity"] is True
assert len(resp["completion"]) == len("The ") + 30
assert set(report) >= {"artifact", "architecture", "params", "training",
                       "adaptation", "optimization", "serving", "limitations"}

## You built a language model

Take a second with this. You started from raw text and a blank file, and you walked
the entire path: **tokenize** the text, **build** a decoder-only transformer from
attention up, **train** and debug it past a real baseline, **modernize** it with the
components in today's frontier models, **adapt** it cheaply with LoRA, **optimize**
it with quantization and a KV cache, and **serve** it behind an API, all backed by an
engineering report that makes it reproducible. Nothing here was a black box you
imported. You wrote the pieces, and now you understand what every line of a real LLM
stack is doing.

What changes from here is only scale and polish, not the ideas:

- **Bigger, cleaner data.** Swap the 1MB slice for a large, deduplicated corpus. The
  same training loop just runs longer and learns more.
- **A real tokenizer.** Replace the character tokenizer with byte-pair encoding (BPE)
  so the model works in subword units, the single biggest quality jump for the least
  architectural change.
- **A GPU run.** Move the loop onto a GPU, widen the config (more layers, heads, and
  embedding width), and lengthen the context. Everything you built here scales up
  unchanged.

The gap between this notebook and a model you have actually heard of is mostly data,
compute, and patience. You now have the map. Go build something bigger.